# Warm-up: Worked Solutions to the Ten-Minute Diagnostic

Companion notebook to the warm-up diagnostic at the start of the study guide (Kundu, Cohen & Dowling 2016, Appendix B).
For each question: the short answer, the intuition, a step-by-step solution, a check with SymPy or NumPy, and a one-line
takeaway. The short answers are also in Appendix B of the study guide. Try each question before reading its solution.

Notation follows Kundu: cylindrical coordinates $(R,\varphi,z)$ and spherical coordinates $(r,\theta,\varphi)$, with $\theta$
measured from the $+z$ axis (co-latitude).

Before committing: **Restart** the kernel, then **Run All**, then **save**, so the notebook runs from top to bottom.

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from scipy.integrate import quad

FIG_DIR = Path("figures")   # figures worth showing on GitHub
FIG_DIR.mkdir(exist_ok=True)

BLUE, ORANGE = "#2a78d6", "#eb6834"        # categorical slots 1 and 2
RAMP = ["#86b6ef", "#2a78d6", "#104281"]   # ordered series: light -> dark
GRAY = "#52514e"                           # reference curves and annotations

x, y, z, t = sp.symbols("x y z t", real=True)


def check(label, ok):
    """Print a one-line confirmation, or stop the notebook if a check fails."""
    if not ok:
        raise AssertionError(f"check failed: {label}")
    print(f"ok  {label}")

## Q1 (Section 1): A constant reading at a river gauge

> A thermometer fixed at a river gauge reads a constant temperature all day. Does every water parcel passing the gauge keep a
> constant temperature?

**Short answer.** Not necessarily. The gauge measures $\partial T/\partial t$ at a fixed point, which is zero; a parcel
experiences $DT/Dt = u\,\partial T/\partial x$, which is not zero when the temperature varies along the river.

**Intuition.** Picture a conveyor-belt pizza oven with an infrared thermometer aimed at one spot on the belt. It always reads
60 °C, because every pizza has reached 60 °C when it passes that spot, yet every pizza is heating up. A constant reading means
*the next parcel arrives as warm as the last one*, not *each parcel keeps its temperature*. The gauge is an Eulerian
observer; a parcel, or a drifter riding with it, is a Lagrangian observer (Section 1.1).

**Step 1. Translate the words.** "Constant reading" means $\partial T/\partial t = 0$ at the gauge. "Each parcel keeps its
temperature" means $DT/Dt = 0$. The question asks whether the first implies the second.

**Step 2. Link the two rates** with the material derivative (B.1.4), for a river flowing with velocity $u$ along $x$:

$$\frac{DT}{Dt} = \frac{\partial T}{\partial t} + u\,\frac{\partial T}{\partial x}
\qquad\text{(following a parcel = at a fixed point + advection).}$$

**Step 3. Substitute** $\partial T/\partial t = 0$: $\quad DT/Dt = u\,\partial T/\partial x$.

**Step 4. Interpret.** This vanishes only if $u = 0$ or $\partial T/\partial x = 0$. Real rivers have along-stream temperature
gradients (a cold release from a reservoir warming downstream, tributaries, shading), so in general $DT/Dt \neq 0$. Each parcel
warms on its way, but colder water keeps arriving from upstream; at the gauge the two effects cancel exactly
(figure below, panel a).

**The reverse case** is Q2: a slug of warm water carried past the gauge without mixing makes the reading rise and fall
($\partial T/\partial t \neq 0$) while every parcel keeps its temperature ($DT/Dt = 0$).

**Takeaway.** Steady at a point ($\partial/\partial t = 0$) is not steady for a parcel ($D/Dt = 0$); the difference is the
advective term $u\,\partial T/\partial x$.

*My notes:*

In [ ]:
# Cold water released from a dam warms downstream by 0.2 degC per km; the river flows at 1 m/s.
u_kmh = 1.0 * 3.6               # 1 m/s in km/h
dTdx = 0.2                      # degC per km
dTdt_gauge = 0.0                # the gauge reading is constant
DTDt = dTdt_gauge + u_kmh * dTdx
print(f"each parcel warms at DT/Dt = {DTDt:.2f} degC/h, while the gauge reading stays constant")

# The same statement with SymPy: a steady field T(x) and a parcel that passes the gauge (x = 0) at t = 0
T0, G, u_river = sp.symbols("T_0 G u", positive=True)
T_field = T0 + G * x                       # steady: no t in it
check("dT/dt = 0 at any fixed point", sp.diff(T_field, t) == 0)
check("the parcel's temperature changes at u G", sp.diff(T_field.subs(x, u_river * t), t) == u_river * G)

## Q2 (Section 1): Solve $\partial f/\partial t + 3\,\partial f/\partial x = 0$ with $f(x,0) = e^{-x^2}$

**Short answer.** $f(x,t) = e^{-(x-3t)^2}$: the initial bell curve moves to the right at speed 3 without changing shape.

**Intuition.** This is Q1 with $Df/Dt = 0$ and $u = 3$: an observer moving at speed 3 sees $f$ stay constant. Picture a patch
of dye in a river flowing at 3 m/s, with no diffusion: the patch is simply carried along.

**Step 1. The total derivative along a path** $x = X(t)$ (Section 1.3):

$$\frac{d}{dt}f\big(X(t),t\big) = \frac{\partial f}{\partial t} + \frac{dX}{dt}\,\frac{\partial f}{\partial x}.$$

**Step 2. Choose paths with $dX/dt = 3$.** Then the right-hand side is the left-hand side of the PDE, which is zero:
$f$ does not change along these paths.

**Step 3. Solve for the paths** (the characteristics): $X(t) = x_0 + 3t$, a family of parallel straight lines in the
$x$–$t$ plane; $x_0$ is the starting point at $t = 0$.

**Step 4. Along each line $f$ keeps its starting value:** $f = f(x_0,0) = e^{-x_0^2}$.

**Step 5. Trace back.** The line through a point $(x,t)$ started at $x_0 = x - 3t$.

**Step 6. Hence** $f(x,t) = e^{-(x-3t)^2}$.

**Check by hand.** With $s = x - 3t$: $\partial f/\partial t = 6s\,e^{-s^2}$ and $\partial f/\partial x = -2s\,e^{-s^2}$, so
$\partial f/\partial t + 3\,\partial f/\partial x = 0$; at $t = 0$, $f = e^{-x^2}$. The peak sits where $x - 3t = 0$: at $x = 6$
when $t = 2$, and $f(7,2) = e^{-1} = f(1,0)$, so the shape is unchanged.

**Pitfall.** $e^{-(x+3t)^2}$ moves to the *left*. Check the direction by following the peak: $x - 3t = 0$ gives $x = 3t$, which
increases with time. In general $F(x - ct)$ moves toward $+x$ at speed $c$.

**Takeaway.** A first-order advection equation translates the initial profile; $f$ is constant along the characteristics
$x - 3t = \text{const}$. Knickpoint retreat and dune migration (Chapters 1 and 2) are the same problem.

*My notes:*

In [ ]:
f_sol = sp.exp(-(x - 3 * t) ** 2)
check("f_t + 3 f_x = 0", sp.simplify(sp.diff(f_sol, t) + 3 * sp.diff(f_sol, x)) == 0)
check("f(x, 0) = exp(-x^2)", sp.simplify(f_sol.subs(t, 0) - sp.exp(-x**2)) == 0)
check("the peak is at x = 6 when t = 2", f_sol.subs({x: 6, t: 2}) == 1)
check("the shape is unchanged: f(7, 2) = f(1, 0)", f_sol.subs({x: 7, t: 2}) == f_sol.subs({x: 1, t: 0}))

wrong = sp.exp(-(x + 3 * t) ** 2)
print("the wrong sign, exp(-(x + 3t)^2), leaves a residual of",
      sp.factor(sp.simplify((sp.diff(wrong, t) + 3 * sp.diff(wrong, x)) / wrong)), "times f")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))

# (a) Q1: steady river temperature T(x) = 15 + 0.2 x (x in km downstream of the gauge), u = 3.6 km/h
hours = np.linspace(-2, 2, 50)
axes[0].plot(hours, np.full_like(hours, 15.0), color=ORANGE, lw=2, label="gauge (fixed point): dT/dt = 0")
for t_pass, color in zip((-1, 0, 1), RAMP):
    axes[0].plot(hours, 15.0 + 0.2 * 3.6 * (hours - t_pass), color=color, lw=2,
                 label=f"parcel passing the gauge at t = {t_pass} h")
    axes[0].plot(t_pass, 15.0, "o", color=color, ms=7)
axes[0].set(xlabel="time (h)", ylabel="temperature (°C)", ylim=(12.6, 18.4),
            title="(a) Q1: a steady reading, yet every parcel warms\n(DT/Dt = u dT/dx = 0.72 °C/h)")
axes[0].legend(fontsize=8, loc="upper left")

# (b) Q2: the profile at three times
xs = np.linspace(-3, 10, 600)
for t_val, color in zip((0, 1, 2), RAMP):
    axes[1].plot(xs, np.exp(-(xs - 3 * t_val) ** 2), color=color, lw=2, label=f"t = {t_val}")
axes[1].set(xlabel="x", ylabel="f(x, t)", ylim=(0, 1.15), title="(b) Q2: the profile translates at speed 3")
axes[1].legend(fontsize=8, loc="upper right")

# (c) Q2: f is constant along the characteristics x - 3t = const
X, Tm = np.meshgrid(np.linspace(-4, 10, 400), np.linspace(0, 2, 200))
cf = axes[2].contourf(X, Tm, np.exp(-(X - 3 * Tm) ** 2), levels=np.linspace(0, 1, 11), cmap="Blues")
for x0 in range(-10, 11):
    axes[2].plot([x0, x0 + 6], [0, 2], color=GRAY, lw=0.7)
axes[2].plot([0, 6], [0, 2], color=ORANGE, lw=2.5, label="characteristic through the peak: x = 3t")
axes[2].set(xlim=(-4, 10), ylim=(0, 2), xlabel="x", ylabel="t",
            title="(c) Q2: f is constant along each line x - 3t = const")
axes[2].legend(fontsize=8, loc="upper left")
fig.colorbar(cf, ax=axes[2], label="f(x, t)")
fig.tight_layout()
fig.savefig(FIG_DIR / "ch00_moving_observers.png", dpi=200, bbox_inches="tight")
plt.show()

## Q3 (Section 2): Derivatives in a moving frame

> With $\xi = x - Ut$ and $\tau = t$, express $\partial/\partial t$ (at fixed $x$) and $\partial/\partial x$ (at fixed $t$) in
> terms of $\partial/\partial\xi$ and $\partial/\partial\tau$.

**Short answer** (Example B.2):

$$\left(\frac{\partial}{\partial t}\right)_x = \frac{\partial}{\partial\tau} - U\,\frac{\partial}{\partial\xi},\qquad
\left(\frac{\partial}{\partial x}\right)_t = \frac{\partial}{\partial\xi}.$$

**Intuition.** $\xi = x - Ut$ is a seat number on a train moving at speed $U$: fixed $\xi$ means sitting still on the train,
and $\tau = t$ is the same clock. So $(\partial/\partial t)_x$ is the rate of change seen by someone standing on the platform,
and $(\partial/\partial\tau)_\xi$ the rate seen by a passenger. Although $\tau = t$, $\partial/\partial\tau \neq \partial/\partial t$,
because a different variable is held fixed. As Kundu remarks, the simplest-looking transformation, $\tau = t$, produces the most
complicated derivative.

**Step 1.** Write the same field in both sets of variables: $f(x,t) = F(\xi,\tau)$.

**Step 2. Chain rule** (B.2.1):

$$\left(\frac{\partial f}{\partial t}\right)_x = \frac{\partial F}{\partial\xi}\frac{\partial\xi}{\partial t}
+ \frac{\partial F}{\partial\tau}\frac{\partial\tau}{\partial t},\qquad
\left(\frac{\partial f}{\partial x}\right)_t = \frac{\partial F}{\partial\xi}\frac{\partial\xi}{\partial x}
+ \frac{\partial F}{\partial\tau}\frac{\partial\tau}{\partial x}.$$

**Step 3. Fill in the derivatives of the new variables with respect to the old ones:**

| | $\partial/\partial t$ (fixed $x$) | $\partial/\partial x$ (fixed $t$) |
|---|---|---|
| $\xi = x - Ut$ | $-U$ | $1$ |
| $\tau = t$ | $1$ | $0$ |

**Step 4. Substitute:** $(\partial f/\partial t)_x = -U\,\partial F/\partial\xi + \partial F/\partial\tau$ and
$(\partial f/\partial x)_t = \partial F/\partial\xi$.

**Step 5. Read it the other way round:** $\partial/\partial\tau = \partial/\partial t + U\,\partial/\partial x$. The passenger's
rate is the platform rate plus $U$ times the spatial gradient: the structure of $D/Dt$ in Q1, and exactly $D/Dt$ when $U$ is the
flow velocity. For example, $f = x + 2t$ with $U = 3$ becomes $F = \xi + 5\tau$: the platform sees $f$ grow at 2, the passenger
at 5 (2 from time itself, plus 3 × 1 from riding across the gradient).

**Use: Q2 in two lines.** With $U = 3$, $\partial f/\partial t + 3\,\partial f/\partial x = 0$ becomes
$(\partial F/\partial\tau - 3\,\partial F/\partial\xi) + 3\,\partial F/\partial\xi = \partial F/\partial\tau = 0$, so $F$
depends on $\xi$ only: $f = F(x - 3t)$, and the initial condition gives $F(x) = e^{-x^2}$. In a frame that moves with the
pattern, nothing changes.

**Pitfall.** For $f = \sin(x - Ut)$, $(\partial f/\partial t)_x = -U\cos(x - Ut)$ but $(\partial F/\partial\tau)_\xi = 0$.

**Takeaway.** After a change of variables, a partial derivative means nothing until you say which variable is held fixed:
write the subscripts.

*My notes:*

In [ ]:
U = sp.Symbol("U", real=True)
xi, tau = sp.symbols("xi tau", real=True)
to_new = {x: xi + U * tau, t: tau}               # old variables in terms of the new ones


def in_new(expr):
    """Rewrite a field f(x, t) as F(xi, tau)."""
    return expr.subs(to_new, simultaneous=True)


# A deliberately messy test field
f_test = sp.exp(-x**2) * sp.cos(t) + x**3 * t**2
F_test = in_new(f_test)
check("(df/dt)_x = dF/dtau - U dF/dxi",
      sp.simplify(in_new(sp.diff(f_test, t)) - (sp.diff(F_test, tau) - U * sp.diff(F_test, xi))) == 0)
check("(df/dx)_t = dF/dxi", sp.simplify(in_new(sp.diff(f_test, x)) - sp.diff(F_test, xi)) == 0)

# The worked numbers: f = x + 2t with U = 3
F_simple = in_new(x + 2 * t).subs(U, 3)
print("x + 2t in the moving frame (U = 3):", F_simple)
check("platform rate 2 = passenger rate 5 - 3 x gradient 1", sp.diff(F_simple, tau) - 3 * sp.diff(F_simple, xi) == 2)

# Q2 in the moving frame: the PDE collapses to dF/dtau = 0
F_gen = sp.Function("F")(xi, tau)
print("Q2 in the moving frame:", sp.Eq((sp.diff(F_gen, tau) - 3 * sp.diff(F_gen, xi)) + 3 * sp.diff(F_gen, xi), 0))

# The pitfall: two different "time derivatives" of sin(x - Ut)
f_wave = sp.sin(x - U * t)
print("(df/dt)_x    =", sp.diff(f_wave, t), "  (cos is even: this is -U cos(x - Ut))")
print("(dF/dtau)_xi =", sp.diff(in_new(f_wave), tau))

## Q4 (Section 2): A similarity variable for a diffusing step

> Which combination of $x$ and $t$ turns $\partial c/\partial t = D\,\partial^2 c/\partial x^2$ (infinite domain, step initial
> condition) into an ODE?

**Short answer.** $\eta = x/\sqrt{Dt}$, conventionally $\eta = x/\sqrt{4Dt}$ (any constant multiple of $x/\sqrt t$ works).
The PDE becomes $F'' + 2\eta F' = 0$, and the solution is an error function.

**Intuition.** Start with dye at concentration $c_0$ in the left half of an infinitely long tube and clear water in the right
half. The transition zone widens with time, roughly as $\sqrt{Dt}$. The problem has no other length scale (the tube is infinite
and the step has no width), so profiles at different times can only be stretched copies of one curve. Measured with the ruler
$\sqrt{Dt}$, they all coincide (the similarity-collapse figure of Chapter 2).

**Method A: dimensions (fastest).** $x$ is in m, $t$ in s and $D$ in m²/s. The only length that $D$ and $t$ can form is
$\sqrt{Dt}$, so $c/c_0$ can depend on $x$ and $t$ only through $x/\sqrt{Dt}$.

**Method B: a trial exponent (Example B.3), which proves it.**

**Step 1.** Try $c = F(\eta)$ with $\eta = x\,t^{\alpha}$. The chain rule gives
$\partial c/\partial t = (\alpha/t)\,\eta F'$, $\;\partial c/\partial x = t^{\alpha}F'$ and $\;\partial^2c/\partial x^2 = t^{2\alpha}F''$.

**Step 2. Substitute:** $(\alpha/t)\,\eta F' = D\,t^{2\alpha}F''$.

**Step 3.** For an equation in $\eta$ alone, $t$ must cancel: the power is $-1$ on the left and $2\alpha$ on the right, so
$2\alpha = -1$, $\alpha = -1/2$, and $\eta \propto x/\sqrt t$.

**Step 4. Scale it as $\eta = x/\sqrt{4Dt}$**, so that $\partial\eta/\partial t = -\eta/(2t)$ and
$\partial\eta/\partial x = 1/\sqrt{4Dt}$. Then $\partial c/\partial t = -\eta F'/(2t)$ and $D\,\partial^2c/\partial x^2 = F''/(4t)$, and
multiplying by $4t$:

$$F'' + 2\eta F' = 0 .$$

The factor 4 is chosen to make this ODE as clean as possible.

**Step 5. Solve it** with $G = F'$: $G' + 2\eta G = 0$ gives $G = C_1e^{-\eta^2}$, so $F = C_1'\,\mathrm{erf}(\eta) + C_2$,
where $\mathrm{erf}(\eta) = \frac{2}{\sqrt\pi}\int_0^\eta e^{-s^2}ds$ and $\mathrm{erf}(\pm\infty) = \pm1$.

**Step 6. The conditions collapse** (this is why the method works). As $t \to 0^+$, $\eta \to +\infty$ for $x > 0$ and
$\eta \to -\infty$ for $x < 0$; as $x \to \pm\infty$ at fixed $t$, $\eta \to \pm\infty$ too. The initial condition and both
boundary conditions become just two conditions, $F(+\infty) = 0$ and $F(-\infty) = c_0$: exactly what a second-order ODE needs.
Hence

$$c(x,t) = \frac{c_0}{2}\left[1 - \mathrm{erf}\left(\frac{x}{\sqrt{4Dt}}\right)\right]
= \frac{c_0}{2}\,\mathrm{erfc}\left(\frac{x}{\sqrt{4Dt}}\right).$$

At $x = 0$, $c = c_0/2$ at all times.

**When it fails.** A finite domain of length $L$, or a block of width $w$ instead of a step, adds a second length scale; the
solution then depends on $L/\sqrt{Dt}$ (or $w/\sqrt{Dt}$) as well, and $\eta$ alone is not enough.
**A detail.** $x^2/(Dt)$ is also dimensionless, but it cannot tell $x$ from $-x$, and the two sides of the step differ.

**Takeaway.** With no intrinsic length scale, the only length is $\sqrt{Dt}$; $\eta = x/\sqrt{4Dt}$ turns the PDE into an ODE
whose solution is an error function.

*My notes:*

In [ ]:
D = sp.Symbol("D", positive=True)
tp = sp.Symbol("t", positive=True)               # time is positive in the diffusion problem
alpha = sp.Symbol("alpha", real=True)
eta = sp.Symbol("eta", real=True)
F = sp.Function("F")

# Steps 1-3: with eta = x t^alpha, both sides must carry the same power of t
print("alpha =", sp.solve(sp.Eq(-1, 2 * alpha), alpha))

# Step 4: substitute c = F(x / sqrt(4 D t)), multiply by 4t and write x in terms of eta
c_trial = F(x / sp.sqrt(4 * D * tp))
residual = 4 * tp * (sp.diff(c_trial, tp) - D * sp.diff(c_trial, x, 2))
ode = sp.simplify(residual.subs(x, eta * sp.sqrt(4 * D * tp)))
print("4t (c_t - D c_xx) =", ode, "  -> F'' + 2 eta F' = 0")
check("the ODE is F'' + 2 eta F' = 0",
      sp.simplify(ode + (F(eta).diff(eta, 2) + 2 * eta * F(eta).diff(eta))) == 0)

# Step 5: solve the ODE
print(sp.dsolve(F(eta).diff(eta, 2) + 2 * eta * F(eta).diff(eta), F(eta)))

# Step 6: the erfc solution and its conditions
c0 = sp.Symbol("c_0", positive=True)
c_step = c0 / 2 * sp.erfc(x / sp.sqrt(4 * D * tp))
check("c satisfies the diffusion equation", sp.simplify(sp.diff(c_step, tp) - D * sp.diff(c_step, x, 2)) == 0)
check("c -> c0 as x -> -oo", sp.limit(c_step, x, -sp.oo) == c0)
check("c -> 0 as x -> +oo", sp.limit(c_step, x, sp.oo) == 0)
check("c = c0/2 at x = 0 for all t", c_step.subs(x, 0) == c0 / 2)
check("initial step: c -> c0 at x = -1 as t -> 0+", sp.limit(c_step.subs(x, -1), tp, 0, "+") == c0)
check("initial step: c -> 0 at x = +1 as t -> 0+", sp.limit(c_step.subs(x, 1), tp, 0, "+") == 0)

# The diffusion length of a degrading fault scarp (Chapter 2): kappa = 0.01 m^2/yr
for years in (1e4, 4e4):
    print(f"sqrt(4 kappa t) after {years:,.0f} yr: {np.sqrt(4 * 0.01 * years):.0f} m")

## Q5 (Section 3): $\nabla\cdot(\nabla\times\mathbf u)$ and $\nabla\times(\nabla\phi)$

**Short answer.** Both vanish for smooth fields (continuous second derivatives): *a curl has no divergence, and a gradient
has no curl* (B.3.11 and B.3.12).

**Method 1: write out the components.** With
$\nabla\times\mathbf u = (\partial_y u_z - \partial_z u_y,\ \partial_z u_x - \partial_x u_z,\ \partial_x u_y - \partial_y u_x)$,

$$\nabla\cdot(\nabla\times\mathbf u) = (\partial_x\partial_y u_z - \partial_y\partial_x u_z)
+ (\partial_z\partial_x u_y - \partial_x\partial_z u_y) + (\partial_y\partial_z u_x - \partial_z\partial_y u_x) = 0 .$$

The six terms cancel in pairs because mixed partial derivatives do not depend on the order of differentiation (Schwarz's
theorem). Likewise, the $z$ component of $\nabla\times\nabla\phi$ is $\partial_x\partial_y\phi - \partial_y\partial_x\phi = 0$,
and so are the other two.

**Method 2: index notation (Section 3.2).**
$\nabla\cdot(\nabla\times\mathbf u) = \partial_i(\varepsilon_{ijk}\partial_j u_k) = \varepsilon_{ijk}\,\partial_i\partial_j u_k$.
Here $\varepsilon_{ijk}$ changes sign when $i$ and $j$ are swapped (antisymmetric), while $\partial_i\partial_j$ does not
(symmetric). For any symmetric $A_{ij}$, renaming the dummy indices gives
$S = \varepsilon_{ijk}A_{ij} = \varepsilon_{jik}A_{ji} = -\varepsilon_{ijk}A_{ij} = -S$, so $S = 0$. The same argument on $j$ and
$k$ gives $(\nabla\times\nabla\phi)_i = \varepsilon_{ijk}\partial_j\partial_k\phi = 0$.

**Two examples.** $\mathbf u = (y^2, z^2, x^2)$ has $\nabla\times\mathbf u = (-2z, -2x, -2y)$, whose divergence is
$0 + 0 + 0$. $\phi = x^2y$ has $\nabla\phi = (2xy, x^2, 0)$, whose curl has $z$ component $2x - 2x = 0$.

**Physical meaning.**

- $\nabla\cdot\boldsymbol\omega = 0$, with $\boldsymbol\omega = \nabla\times\mathbf u$: vorticity has no sources or sinks, so
  vortex lines cannot start or end inside the fluid. They close on themselves (smoke rings) or end at boundaries (a tornado on
  the ground, a bathtub vortex at the free surface and the drain).
- $\nabla\times\nabla\phi = 0$: gradient fields do not rotate. With Stokes' theorem, $\oint\nabla\phi\cdot d\mathbf l = 0$:
  walk around any closed loop on a hill and your net climb is zero, which is why Escher's endless staircase
  (*Ascending and Descending*) cannot exist in a real landscape. In fluids, potential flow $\mathbf u = \nabla\Phi$ is
  irrotational, and so is Darcy flow $\mathbf q = -K\nabla h$ with uniform $K$.

**Takeaway.** Both identities follow from two facts: mixed partial derivatives commute, and $\varepsilon_{ijk}$ is
antisymmetric. Q10 shows what happens when the domain has a hole.

*My notes:*

In [ ]:
def grad3(s):
    return [sp.diff(s, v) for v in (x, y, z)]


def div3(v):
    return sp.diff(v[0], x) + sp.diff(v[1], y) + sp.diff(v[2], z)


def curl3(v):
    return [sp.diff(v[2], y) - sp.diff(v[1], z),
            sp.diff(v[0], z) - sp.diff(v[2], x),
            sp.diff(v[1], x) - sp.diff(v[0], y)]


u_gen = [sp.Function(name)(x, y, z) for name in ("u_x", "u_y", "u_z")]   # any smooth field
phi_gen = sp.Function("phi")(x, y, z)
check("div(curl u) = 0 for a general field", sp.simplify(div3(curl3(u_gen))) == 0)
check("curl(grad phi) = 0 for a general field", all(sp.simplify(c) == 0 for c in curl3(grad3(phi_gen))))

# Index notation: sum_ij eps_ijk A_ij = 0 for any symmetric A
a_ = sp.symbols("a0:6")
A_sym = sp.Matrix([[a_[0], a_[1], a_[2]], [a_[1], a_[3], a_[4]], [a_[2], a_[4], a_[5]]])
for k in range(3):
    S = sum(sp.LeviCivita(i, j, k) * A_sym[i, j] for i in range(3) for j in range(3))
    check(f"sum_ij eps_ij{k + 1} A_ij = 0 for a symmetric A", S == 0)

# The two worked examples
v_ex = [y**2, z**2, x**2]
print("curl(y^2, z^2, x^2) =", curl3(v_ex))
check("its divergence is 0", div3(curl3(v_ex)) == 0)
print("grad(x^2 y) =", grad3(x**2 * y))
check("its curl is 0", curl3(grad3(x**2 * y)) == [0, 0, 0])

## Q6 (Section 3): Gauss' divergence theorem and the meaning of $\nabla\cdot\mathbf q$

**Short answer.** For a volume $V$ enclosed by a surface $A$ with outward unit normal $\mathbf n$,

$$\oint_A \mathbf q\cdot\mathbf n\,dA = \int_V \nabla\cdot\mathbf q\,dV .$$

The left side is the net rate at which the flux $\mathbf q$ (of water, heat, sediment...) leaves through the boundary (outflow
positive, inflow negative); the right side adds up $\nabla\cdot\mathbf q$ over the interior. Shrinking $V$ to a point gives
$\nabla\cdot\mathbf q = \lim_{V\to0}V^{-1}\oint_A\mathbf q\cdot\mathbf n\,dA$: the net outflow per unit volume.

**Method 1: shrink the volume.** Around a point $P$, a small volume $\Delta V$ has nearly uniform $\nabla\cdot\mathbf q$, so
$\int_V\nabla\cdot\mathbf q\,dV \approx (\nabla\cdot\mathbf q)_P\,\Delta V$. Dividing by $\Delta V$, $(\nabla\cdot\mathbf q)_P$ is
literally "net outflow ÷ volume". This is the coordinate-free definition of the divergence.

**Method 2: book-keeping on a small box** $dx\times dy\times dz$.

- Right face (at $x + dx$, outward normal $+\mathbf e_x$): outflow $q_x(x + dx)\,dy\,dz$.
- Left face (at $x$, outward normal $-\mathbf e_x$): outflow $-q_x(x)\,dy\,dz$, i.e. an inflow $q_x(x)\,dy\,dz$.
- Net outflow through this pair of faces: $[q_x(x + dx) - q_x(x)]\,dy\,dz \approx (\partial q_x/\partial x)\,dx\,dy\,dz$.

Adding the $y$ and $z$ pairs gives a net outflow $(\nabla\cdot\mathbf q)\,dV$. What matters is not the size of $\mathbf q$
but how much it changes across the box: if as much enters on one side as leaves on the other, nothing accumulates, just as the
number of people in a room changes only when the number leaving differs from the number entering.

**Why the theorem holds.** Cut $V$ into many small boxes and add their net outflows. On every interior face, the outflow from
one box is the inflow to its neighbour, so these cancel in pairs; only the outer boundary is left.

**Example.** $\mathbf q = (x, y, z)$ has $\nabla\cdot\mathbf q = 3$. On the unit cube, the faces at $x$, $y$, $z = 1$ each pass
an outflow of 1 and the faces at 0 pass nothing: a net outflow of 3, equal to $3 \times 1$. On a sphere of radius $R$,
$\mathbf q\cdot\mathbf n = R$, so the outflow is $R \times 4\pi R^2 = 4\pi R^3 = 3 \times \tfrac43\pi R^3$.

**Signs.** $\nabla\cdot\mathbf q > 0$: divergence, a source (more leaves than enters). $\nabla\cdot\mathbf q < 0$:
convergence, a sink.

**Research connection.** In the Exner equation $(1-\lambda_p)\,\partial z_b/\partial t + \nabla\cdot\mathbf q_s = 0$, the bed
lowers where the sediment flux diverges and rises where it converges. With $\mathbf q = -K\nabla z$ it gives
$\partial z/\partial t = K\nabla^2z$, so convex hilltops ($\nabla^2z < 0$) erode (Chapter 3).

**Takeaway.** Divergence is net outflow per unit volume. Gauss' theorem adds up the net outflows of many small boxes: the
interior faces cancel and only the outer boundary remains.

*My notes:*

In [ ]:
# (1) q = (x, y, z) on the unit cube and on a sphere
q = [x, y, z]
cube_out = (sp.integrate(q[0].subs(x, 1) - q[0].subs(x, 0), (y, 0, 1), (z, 0, 1))
            + sp.integrate(q[1].subs(y, 1) - q[1].subs(y, 0), (x, 0, 1), (z, 0, 1))
            + sp.integrate(q[2].subs(z, 1) - q[2].subs(z, 0), (x, 0, 1), (y, 0, 1)))
cube_vol = sp.integrate(div3(q), (x, 0, 1), (y, 0, 1), (z, 0, 1))
check(f"unit cube: net outflow {cube_out} = volume integral {cube_vol}", cube_out == cube_vol)
Rs = sp.Symbol("R", positive=True)
check("sphere: R x 4 pi R^2 = 3 x (4/3) pi R^3",
      sp.simplify(Rs * 4 * sp.pi * Rs**2 - 3 * sp.Rational(4, 3) * sp.pi * Rs**3) == 0)

# (2) Net outflow / volume of a shrinking cube centred on P tends to div q at P
h = sp.Symbol("h", positive=True)                     # half-width of the cube
q2 = [sp.sin(x), y**2, x * z]
P = {x: sp.Rational(3, 10), y: sp.Rational(1, 2), z: sp.Rational(1, 5)}
span = {v: (P[v] - h, P[v] + h) for v in (x, y, z)}


def face_pair(component, var, others):
    """Outflow through the two faces normal to `var`."""
    (a1, b1), (a2, b2) = span[others[0]], span[others[1]]
    jump = component.subs(var, span[var][1]) - component.subs(var, span[var][0])
    return sp.integrate(jump, (others[0], a1, b1), (others[1], a2, b2))


outflow = face_pair(q2[0], x, (y, z)) + face_pair(q2[1], y, (x, z)) + face_pair(q2[2], z, (x, y))
per_volume = outflow / (2 * h) ** 3
print("net outflow / volume for half-widths 0.1, 0.01, 0.001:",
      [round(float(per_volume.subs(h, hv)), 6) for hv in (0.1, 0.01, 0.001)])
div_P = div3(q2).subs(P)
print(f"div q at P = {div_P} = {float(div_P):.6f}")
check("limit of net outflow / volume = div q at P", sp.simplify(sp.limit(per_volume, h, 0) - div_P) == 0)

# (3) Exner: a 100 m reach where the unit-width sediment flux rises from 1.00 to 1.05 m^2/day
q_in, q_out, reach, porosity = 1.00, 1.05, 100.0, 0.4
print(f"net outflow per unit bed area: {(q_out - q_in) / reach * 1e3:.2f} mm/day; "
      f"bed lowering with porosity 0.4: {(q_out - q_in) / reach / (1 - porosity) * 1e3:.2f} mm/day")

## Q7 (Section 4): Three small questions on the delta function

> Evaluate $\int_{-\infty}^{\infty}e^{x}\,\delta(x-2)\,dx$ and $\int_{-\infty}^{\infty}\delta(3x)\,dx$. If $x$ is in metres,
> what are the units of $\delta(x)$?

**Short answers.** $e^2 \approx 7.389$; $\;1/3$; $\;\mathrm{m^{-1}}$.

**(a) Sifting (B.4.4).** $\int f(x)\,\delta(x - x_0)\,dx = f(x_0)$: the unit-area spike at $x = 2$ picks out the value of the
rest of the integrand there, so the integral is $e^2$. To see why, replace $\delta$ by a box $\delta_\varepsilon(x - 2)$ of
width $\varepsilon$ and height $1/\varepsilon$:

$$\int e^x\,\delta_\varepsilon(x-2)\,dx = \frac1\varepsilon\int_{2-\varepsilon/2}^{2+\varepsilon/2}e^x\,dx ,$$

the average of $e^x$ over a window around $x = 2$, which tends to $e^2$ as the window shrinks (figure, panel a). The spike sits
where the argument of $\delta$ vanishes: $x - 2 = 0$, so $x = 2$, not $-2$.

**(b) Scaling.** Substitute $s = 3x$, $dx = ds/3$, with unchanged limits:
$\int\delta(3x)\,dx = \int\delta(s)\,ds/3 = 1/3$. In general $\delta(\alpha x) = \delta(x)/|\alpha|$. With boxes,
$\delta_\varepsilon(3x)$ keeps the height $1/\varepsilon$ but is non-zero only for $|x| < \varepsilon/6$, a width of
$\varepsilon/3$, so its area is $1/3$ (panel b); in fact $\delta_\varepsilon(3x) = \tfrac13\,\delta_{\varepsilon/3}(x)$. For
$\alpha < 0$ the substitution flips the limits; flipping them back gives a minus sign that cancels the sign of $1/\alpha$,
hence $|\alpha|$.

**(c) Units.** $\int\delta(x)\,dx = 1$ is a pure number and $dx$ is in metres, so $\delta(x)$ is in $\mathrm{m^{-1}}$. In
general $[\delta(x)] = 1/[x]$, and in three dimensions $\delta(\mathbf x) = \delta(x)\,\delta(y)\,\delta(z)$ is in
$\mathrm{m^{-3}}$. Check with a tracer test: dumping $M = 10$ kg of salt into a channel of cross-section $A = 20\ \mathrm{m^2}$
gives $c(x,0) = (M/A)\,\delta(x)$, in $(\mathrm{kg\,m^{-2}})(\mathrm{m^{-1}}) = \mathrm{kg\,m^{-3}}$, a concentration. Likewise
the rainfall impulse $\delta(t)$ of an instantaneous unit hydrograph is in $\mathrm{s^{-1}}$.

**Takeaway.** $\delta$ is a unit-area spike of zero width: it samples the integrand where its argument vanishes, squeezing
it by a factor $\alpha$ divides its area by $|\alpha|$, and its units are the inverse of its argument's.

*My notes:*

In [ ]:
check("sifting: int e^x delta(x - 2) dx = e^2",
      sp.integrate(sp.exp(x) * sp.DiracDelta(x - 2), (x, -sp.oo, sp.oo)) == sp.exp(2))
check("scaling: int delta(3x) dx = 1/3", sp.integrate(sp.DiracDelta(3 * x), (x, -sp.oo, sp.oo)) == sp.Rational(1, 3))
check("a negative factor: int delta(-3x) dx = 1/3",
      sp.integrate(sp.DiracDelta(-3 * x), (x, -sp.oo, sp.oo)) == sp.Rational(1, 3))


def box_average(eps):
    """Integral of e^x times a box of width eps and height 1/eps centred on x = 2."""
    return (np.exp(2 + eps / 2) - np.exp(2 - eps / 2)) / eps


for eps in (1.0, 0.1, 0.01):
    print(f"box width {eps:4.2f}: {box_average(eps):.5f}   (e^2 = {np.exp(2):.5f})")

M_salt, A_channel = 10.0, 20.0
print(f"tracer test: c(x, 0) = {M_salt / A_channel} kg/m^2 x delta(x) [1/m] -> a concentration in kg/m^3")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))

xs = np.linspace(0.8, 3.0, 400)
axes[0].plot(xs, np.exp(xs), color=GRAY, lw=1.5, label="e^x")
for eps, color in zip((1.0, 0.5, 0.2), RAMP):
    avg = box_average(eps)
    axes[0].hlines(avg, 2 - eps / 2, 2 + eps / 2, color=color, lw=3,
                   label=f"average over a window of width {eps:g}: {avg:.3f}")
axes[0].plot(2, np.exp(2), "o", color=ORANGE, ms=8, label=f"e^2 = {np.exp(2):.3f}")
axes[0].set(xlim=(1.3, 2.7), ylim=(5.0, 11.0), xlabel="x", ylabel="value of e^x",
            title="(a) sifting: the window average tends to e^2")
axes[0].legend(fontsize=8, loc="upper left")

eps = 1.0
xb = np.linspace(-0.8, 0.8, 1601)
box_x = np.where(np.abs(xb) < eps / 2, 1 / eps, 0.0)
box_3x = np.where(np.abs(3 * xb) < eps / 2, 1 / eps, 0.0)
axes[1].fill_between(xb, box_x, color="#d4e4f7", lw=0)        # light tint of BLUE
axes[1].plot(xb, box_x, color=BLUE, lw=2, label="box delta_eps(x): area 1")
axes[1].fill_between(xb, box_3x, color="#f9d2c2", lw=0)       # light tint of ORANGE
axes[1].plot(xb, box_3x, color=ORANGE, lw=2, label="delta_eps(3x): area 1/3")
axes[1].set(xlabel="x (units of eps)", ylabel="height (1/eps)", ylim=(0, 1.4),
            title="(b) delta(3x): same height, one third of the width")
axes[1].legend(fontsize=8, loc="upper right")
fig.tight_layout()
fig.savefig(FIG_DIR / "ch00_delta.png", dpi=200, bbox_inches="tight")
plt.show()

## Q8 (Section 5): Area on a sphere, and unit vectors that turn

> What is the area element on a sphere of radius $a$? Why is $\partial\mathbf e_R/\partial\varphi \neq 0$?

**Short answer.** $dA = a^2\sin\theta\,d\theta\,d\varphi$. The direction of $\mathbf e_R$ depends on $\varphi$ (it always points
away from the $z$ axis), so $\partial\mathbf e_R/\partial\varphi = \mathbf e_\varphi$.

**(a) The area element.**

**Step 1.** Hold $\varphi$ fixed and change $\theta$ by $d\theta$: you move along a meridian, a great circle of radius $a$,
through an arc $a\,d\theta$.

**Step 2.** Hold $\theta$ fixed and change $\varphi$ by $d\varphi$: you move along a circle of latitude. Its radius is not $a$
but the distance to the $z$ axis, $a\sin\theta$, so the arc is $a\sin\theta\,d\varphi$.

**Step 3.** The two arcs are perpendicular, so $dA = (a\,d\theta)(a\sin\theta\,d\varphi) = a^2\sin\theta\,d\theta\,d\varphi$.

These are the scale factors $h_\theta = r$ and $h_\varphi = r\sin\theta$ at $r = a$: on a coordinate surface, the area element
is the product of the scale factors of the two coordinates that vary on it. The factor $\sin\theta$ appears because circles of
latitude shrink toward the poles: the same $d\varphi$ covers less ground at high latitude, like the segments of an orange.
Check: $\int_0^{2\pi}\!\int_0^\pi a^2\sin\theta\,d\theta\,d\varphi = a^2\cdot2\cdot2\pi = 4\pi a^2$.

**Consequences for gridded data (Chapter 5).** In latitude $\vartheta = 90^\circ - \theta$, $\sin\theta = \cos\vartheta$. A 1° × 1°
cell at 60° latitude has about half the area of one at the equator, and the band between 30°S and 30°N is exactly half of
Earth's surface. Global means of gridded fields must therefore be area-weighted; on a Mercator map Greenland looks as large as
Africa, although Africa is about 14 times larger.

**(b) Turning unit vectors.** In Cartesian components,

$$\mathbf e_R = (\cos\varphi,\ \sin\varphi,\ 0),\qquad \mathbf e_\varphi = (-\sin\varphi,\ \cos\varphi,\ 0).$$

$\mathbf e_R$ has unit length everywhere, but its direction depends on $\varphi$: $(1,0,0)$ at $\varphi = 0$, $(0,1,0)$ at
$\varphi = 90^\circ$. Differentiating, $\partial\mathbf e_R/\partial\varphi = (-\sin\varphi,\cos\varphi,0) = \mathbf e_\varphi$ and
$\partial\mathbf e_\varphi/\partial\varphi = -\mathbf e_R$. The derivative of a vector records changes of direction as well
as length; a unit vector can only turn, so its derivative is perpendicular to it. People sitting round a table all point
"outward", yet their arrows differ; in Cartesian coordinates "east" is the same direction everywhere, so these derivatives
vanish. In spherical coordinates, $\partial\mathbf e_r/\partial\varphi = \sin\theta\,\mathbf e_\varphi$ (zero at the poles).

**Takeaway.** Circles of latitude have radius $a\sin\theta$, hence the $\sin\theta$ in $dA$. Curvilinear unit vectors turn
with position, hence their non-zero derivatives, which are the source of the extra terms in Q9.

*My notes:*

In [ ]:
a_s, th, ph = sp.symbols("a theta varphi", positive=True)
check("integral of a^2 sin(theta) over the sphere = 4 pi a^2",
      sp.integrate(a_s**2 * sp.sin(th), (th, 0, sp.pi), (ph, 0, 2 * sp.pi)) == 4 * sp.pi * a_s**2)

a_earth = 6371.0                                     # km
print(f"Earth's surface area: {4 * np.pi * a_earth**2:.3e} km^2")


def cell_area(lat_deg, size_deg=1.0):
    """Area (km^2) of a latitude-longitude cell whose southern edge is at lat_deg (cos(lat) d lat integrates to sin)."""
    lat1, lat2 = np.radians(lat_deg), np.radians(lat_deg + size_deg)
    return a_earth**2 * np.radians(size_deg) * (np.sin(lat2) - np.sin(lat1))


print(f"1 x 1 degree cell: {cell_area(0):,.0f} km^2 at the equator, {cell_area(60):,.0f} km^2 at 60N "
      f"(ratio {cell_area(60) / cell_area(0):.2f})")
band = (np.sin(np.radians(30)) - np.sin(np.radians(-30))) / 2
check(f"30S-30N holds {band:.2f} of the sphere's area", np.isclose(band, 0.5))

e_R = sp.Matrix([sp.cos(ph), sp.sin(ph), 0])
e_phi = sp.Matrix([-sp.sin(ph), sp.cos(ph), 0])
e_r = sp.Matrix([sp.sin(th) * sp.cos(ph), sp.sin(th) * sp.sin(ph), sp.cos(th)])
check("d e_R / d varphi = e_varphi", sp.simplify(e_R.diff(ph) - e_phi) == sp.zeros(3, 1))
check("d e_varphi / d varphi = -e_R", sp.simplify(e_phi.diff(ph) + e_R) == sp.zeros(3, 1))
check("d e_r / d varphi = sin(theta) e_varphi", sp.simplify(e_r.diff(ph) - sp.sin(th) * e_phi) == sp.zeros(3, 1))
step = np.array([np.cos(0.01), np.sin(0.01), 0.0]) - np.array([1.0, 0.0, 0.0])
print(f"e_R(0.01) - e_R(0) = ({step[0]:.6f}, {step[1]:.6f}, {step[2]:.0f}), about 0.01 e_varphi = (0, 0.01, 0)")

## Q9 (Section 6): Where $-u_\varphi^2/R$ comes from

> Why does the radial Navier–Stokes equation in cylindrical coordinates contain $-u_\varphi^2/R$, although no such term
> appears in Cartesian coordinates?

**Short answer.** It is the centripetal acceleration. It comes from the acceleration $D\mathbf u/Dt$, through the turning of
$\mathbf e_\varphi$ along a particle path: geometry, not a new force. Cartesian unit vectors never turn, so the same physics hides
inside the ordinary terms $(\mathbf u\cdot\nabla)u_x$ and $(\mathbf u\cdot\nabla)u_y$.

**Intuition.** Run round a circular track at a steady 5 m/s. In $(R,\varphi)$ components, $u_R = 0$ and $u_\varphi = 5$:
neither changes. Yet you are accelerating, because your direction keeps changing, and your shoes must push you toward the
centre. Constant components hide that acceleration; $-u_\varphi^2/R$ puts it back.

**Step 1.** Write $\mathbf u = u_R\,\mathbf e_R + u_\varphi\,\mathbf e_\varphi + u_z\,\mathbf e_z$. Newton's law needs the
parcel's acceleration $D\mathbf u/Dt$.

**Step 2. Product rule** on the components *and* the unit vectors:

$$\frac{D\mathbf u}{Dt} = \frac{Du_R}{Dt}\mathbf e_R + u_R\frac{D\mathbf e_R}{Dt} + \frac{Du_\varphi}{Dt}\mathbf e_\varphi
+ u_\varphi\frac{D\mathbf e_\varphi}{Dt} + \frac{Du_z}{Dt}\mathbf e_z .$$

**Step 3. How fast do the unit vectors turn?** They depend only on $\varphi$ (Q8), and a parcel's azimuth changes at
$D\varphi/Dt = u_\varphi/R$ (angular speed = speed/radius), so

$$\frac{D\mathbf e_R}{Dt} = \frac{\partial\mathbf e_R}{\partial\varphi}\frac{D\varphi}{Dt} = \frac{u_\varphi}{R}\,\mathbf e_\varphi,
\qquad \frac{D\mathbf e_\varphi}{Dt} = -\frac{u_\varphi}{R}\,\mathbf e_R .$$

**Step 4. Collect by direction:**

$$\frac{D\mathbf u}{Dt} = \left(\frac{Du_R}{Dt} - \frac{u_\varphi^2}{R}\right)\mathbf e_R
+ \left(\frac{Du_\varphi}{Dt} + \frac{u_Ru_\varphi}{R}\right)\mathbf e_\varphi + \frac{Du_z}{Dt}\,\mathbf e_z ,$$

where $Du_R/Dt = \partial_t u_R + u_R\,\partial_R u_R + (u_\varphi/R)\,\partial_\varphi u_R + u_z\,\partial_z u_R$ is the part
that looks Cartesian. The term $-u_\varphi^2/R$ comes from $u_\varphi\,D\mathbf e_\varphi/Dt$.

**Check.** Uniform circular motion with $R = 2$ m and $u_\varphi = 4$ m/s: $Du_R/Dt = 0$, so the radial acceleration is
$-4^2/2 = -8\ \mathrm{m\,s^{-2}}$ (toward the centre). In Cartesian coordinates the position is $(2\cos2t, 2\sin2t)$, whose
acceleration is $-\Omega^2$ times the position, with magnitude $\Omega^2R = 4 \times 2 = 8\ \mathrm{m\,s^{-2}}$. For rigid
rotation $\mathbf u = (-\Omega y, \Omega x)$, the Cartesian term $(\mathbf u\cdot\nabla)\mathbf u = -\Omega^2(x, y)$ contains
the centripetal acceleration by itself, because $u_x$ and $u_y$ change as a parcel goes round; in cylindrical components
$u_R = 0$ and $u_\varphi = \Omega R$ do not change, and $-u_\varphi^2/R = -\Omega^2R$ supplies it. Same physics, different
book-keeping.

**The companion term** $u_Ru_\varphi/R$ expresses conservation of angular momentum: a parcel moving inward ($u_R < 0$) spins
up, like a skater pulling in her arms or water near a bathtub drain.

**Not a centrifugal force.** Moved to the right-hand side, $+u_\varphi^2/R$ looks like a centrifugal force, but we are still in
an inertial frame: curvilinear coordinates are not a rotating frame of reference.

**Research connection: superelevation in river bends.** In a bend of radius $R_c$, the radial balance
$-u_\varphi^2/R \approx -\rho^{-1}\partial p/\partial R$ with hydrostatic pressure $p = \rho g(\zeta - z)$ gives a transverse
water-surface slope $\partial\zeta/\partial R \approx u_\varphi^2/(gR)$ and a superelevation $\Delta\zeta \approx U^2B/(gR_c)$
across a channel of width $B$. The flow is faster near the surface than near the bed, while the pressure gradient is nearly
uniform with depth, so surface water is pushed outward and near-bed water inward: a helical secondary flow that scours the
outer bank and builds point bars on the inner bank. Tea leaves gather at the centre of a stirred cup for the same reason;
Einstein used this "tea-leaf paradox" in a short 1926 paper to explain why rivers meander.

**Takeaway.** $-u_\varphi^2/R$ is the centripetal acceleration produced by the turning of $\mathbf e_\varphi$: geometry, not
a new force. Cartesian unit vectors do not turn, so there it hides in the ordinary terms.

*My notes:*

In [ ]:
# A particle at radius R(t) and azimuth phi(t): project its Cartesian acceleration onto e_R and e_phi
R_t, phi_t = sp.Function("R")(t), sp.Function("varphi")(t)
position = sp.Matrix([R_t * sp.cos(phi_t), R_t * sp.sin(phi_t)])
acc = position.diff(t, 2)
eR_t, ephi_t = sp.Matrix([sp.cos(phi_t), sp.sin(phi_t)]), sp.Matrix([-sp.sin(phi_t), sp.cos(phi_t)])
a_R, a_phi = sp.simplify(acc.dot(eR_t)), sp.simplify(acc.dot(ephi_t))
print("a . e_R   =", a_R)
print("a . e_phi =", a_phi)

u_R, u_phi = R_t.diff(t), R_t * phi_t.diff(t)      # u_R = dR/dt and u_phi = R dphi/dt along the path
check("a . e_R = du_R/dt - u_phi^2 / R", sp.simplify(a_R - (u_R.diff(t) - u_phi**2 / R_t)) == 0)
check("a . e_phi = du_phi/dt + u_R u_phi / R", sp.simplify(a_phi - (u_phi.diff(t) + u_R * u_phi / R_t)) == 0)

# Rigid rotation in Cartesian components: the centripetal acceleration appears by itself
Om = sp.Symbol("Omega", positive=True)
ux_rigid, uy_rigid = -Om * y, Om * x
convective = [ux_rigid * sp.diff(c, x) + uy_rigid * sp.diff(c, y) for c in (ux_rigid, uy_rigid)]
check("(u . grad) u = -Omega^2 (x, y) for rigid rotation", convective == [-Om**2 * x, -Om**2 * y])

print(f"uniform circular motion, R = 2 m, u_phi = 4 m/s: radial acceleration {-4.0**2 / 2.0:.0f} m/s^2")
U_bend, B_bend, g_acc, R_bend = 1.5, 30.0, 9.81, 100.0
print(f"superelevation for U = 1.5 m/s, B = 30 m, R_c = 100 m: {U_bend**2 * B_bend / (g_acc * R_bend) * 100:.1f} cm")

## Q10 (Section 6): Is the potential vortex rotational?

> Is the flow $u_\varphi = \Gamma/(2\pi R)$, $u_R = u_z = 0$ rotational?

**Short answer.** No, except on the axis. For $R \neq 0$ the vorticity is zero, but the circulation around any loop enclosing
the axis is $\Gamma$, so all the vorticity is concentrated on the axis: $\omega_z = \Gamma\,\delta(x)\,\delta(y)$, a point
vortex.

**First, the concept.** "Rotational" asks whether fluid elements *spin about their own centres*, not whether they *travel in
circles*. A Ferris-wheel gondola travels round a big circle but stays upright: it does not spin, like an element in the potential
vortex. A sticker on a spinning record turns once per revolution, always facing outward, like an element in rigid rotation
$u_\varphi = \Omega R$.

**Step 1. The vorticity.** For $u_R = u_z = 0$ and $u_\varphi(R)$,

$$\omega_z = \frac1R\frac{d(Ru_\varphi)}{dR},\qquad Ru_\varphi = \frac{\Gamma}{2\pi} = \text{const}
\quad\Longrightarrow\quad \omega_z = 0\quad(R \neq 0),$$

and $\omega_R = R^{-1}\partial_\varphi u_z - \partial_z u_\varphi = 0$, $\;\omega_\varphi = \partial_z u_R - \partial_R u_z = 0$.

**Step 2. Why the elements do not spin.** Expand $\omega_z = \partial u_\varphi/\partial R + u_\varphi/R$ and imagine a tiny cross
of two matchsticks, one along the circle and one along the radius:

- $u_\varphi/R$: being carried round the circle turns the tangential matchstick anticlockwise;
- $\partial u_\varphi/\partial R$: with $u_\varphi \propto 1/R$ the inner end moves faster than the outer end, so the shear
  turns the radial matchstick clockwise.

For the potential vortex, $\partial u_\varphi/\partial R = -\Gamma/(2\pi R^2)$ and $u_\varphi/R = +\Gamma/(2\pi R^2)$: the two
effects cancel. The cross is sheared but, on average, does not turn: a tiny vane (a vorticity meter, which turns at
$\omega_z/2$) carried round the vortex keeps pointing the same way (figure, panel b). For rigid rotation, both effects equal
$\Omega$ and add up to $\omega_z = 2\Omega$, and the vane turns once per lap (panel a).

**Step 3. The circulation is not zero.** Around a circle of radius $R_0$,

$$C = \oint\mathbf u\cdot d\mathbf l = \int_0^{2\pi}u_\varphi R_0\,d\varphi = \frac{\Gamma}{2\pi R_0}\,2\pi R_0 = \Gamma ,$$

whatever $R_0$. By Stokes' theorem, $C$ equals the integral of $\omega_z$ over the disc. If $\omega_z$ vanished at the centre too,
$C$ would be zero: a contradiction. So all the vorticity sits on the axis, $\omega_z = \Gamma\,\delta(x)\,\delta(y)$, in units of
$(\mathrm{m^2\,s^{-1}})(\mathrm{m^{-2}}) = \mathrm{s^{-1}}$, as a vorticity should be (the unit rule of Q7). A loop that does not
enclose the axis has zero circulation (panel c).

**A Rankine vortex makes the spike concrete** (Chapter 6): a rigidly rotating core of radius $a$ (vorticity $2\Omega$) inside a
potential vortex, with $\Gamma = 2\Omega\,\pi a^2$. Shrink the core at fixed $\Gamma$, and the core vorticity $\Gamma/(\pi a^2)$
grows without bound while its integral stays $\Gamma$: a delta function, exactly as in Section 4.

**Going further (a link to Q5).** The velocity potential $\Phi = \Gamma\varphi/(2\pi)$ gives
$\nabla\Phi = R^{-1}(\partial\Phi/\partial\varphi)\,\mathbf e_\varphi = \mathbf u$, but it grows by $\Gamma$ on every lap: it is
multi-valued, like the spiral ramp of a car park that takes you up one floor each time round the central pillar. Q5 said a
real landscape cannot hold an endless staircase; that is true in a domain without holes. Here the axis is a hole, which is what
"irrotational except on the axis" means.

**In nature.** The outer parts of bathtub vortices, dust devils, tornadoes and eddies behind obstacles in rivers are close to
potential vortices; their cores are close to rigid rotation (Rankine vortices).

**Takeaway.** Travelling in circles is not rotation. For $R > 0$ the flow is irrotational (elements deform without spinning),
but the circulation $\Gamma$ shows that all the vorticity is concentrated on the axis: $\omega_z = \Gamma\,\delta(x)\,\delta(y)$.

*My notes:*

In [ ]:
Rv = sp.Symbol("R", positive=True)
Gam = sp.Symbol("Gamma", positive=True)
Om = sp.Symbol("Omega", positive=True)
ph_v = sp.Symbol("varphi", real=True)
u_vortex = Gam / (2 * sp.pi * Rv)
check("omega_z = (1/R) d(R u_phi)/dR = 0 for R > 0", sp.simplify(sp.diff(Rv * u_vortex, Rv) / Rv) == 0)
check("the two turning rates cancel: du_phi/dR = -u_phi/R", sp.simplify(sp.diff(u_vortex, Rv) + u_vortex / Rv) == 0)
check("circulation around any circle = Gamma", sp.simplify(sp.integrate(u_vortex * Rv, (ph_v, 0, 2 * sp.pi))) == Gam)
check("rigid rotation: omega_z = 2 Omega", sp.simplify(sp.diff(Rv * Om * Rv, Rv) / Rv) == 2 * Om)

# The same in Cartesian components, away from the axis
ux_v = -Gam / (2 * sp.pi) * y / (x**2 + y**2)
uy_v = Gam / (2 * sp.pi) * x / (x**2 + y**2)
check("Cartesian omega_z = 0 off the axis", sp.simplify(sp.diff(uy_v, x) - sp.diff(ux_v, y)) == 0)

# Circulation around two squares, by numerical line integrals (Gamma = 1)
vx = sp.lambdify((x, y), ux_v.subs(Gam, 1))
vy = sp.lambdify((x, y), uy_v.subs(Gam, 1))


def square_circulation(x0, x1, y0, y1):
    """Anticlockwise line integral of u . dl around the rectangle [x0, x1] x [y0, y1]."""
    bottom = quad(lambda s: vx(s, y0), x0, x1)[0]
    right = quad(lambda s: vy(x1, s), y0, y1)[0]
    top = quad(lambda s: vx(s, y1), x0, x1)[0]
    left = quad(lambda s: vy(x0, s), y0, y1)[0]
    return bottom + right - top - left


C_around = square_circulation(-1, 1, -1, 1)
C_away = square_circulation(0.6, 1.6, 0.6, 1.6)
check(f"square around the axis: C = {C_around:.6f} Gamma", np.isclose(C_around, 1.0))
check(f"square away from the axis: C = {abs(C_away):.6f}", np.isclose(C_away, 0.0, atol=1e-10))

for a_core in (0.1, 0.01, 0.001):
    print(f"Rankine core of radius {a_core:g} m with Gamma = 1 m^2/s: core vorticity {1 / (np.pi * a_core**2):,.1f} 1/s")

In [ ]:
def draw_cross(ax, cx, cy, angle, size=0.17):
    """A small vane: one arm at `angle` (blue) and one perpendicular to it (orange)."""
    for extra, color in ((0.0, BLUE), (np.pi / 2, ORANGE)):
        dx, dy = size * np.cos(angle + extra), size * np.sin(angle + extra)
        ax.plot([cx - dx, cx + dx], [cy - dy, cy + dy], color=color, lw=2.5, solid_capstyle="round")


fig, axes = plt.subplots(1, 3, figsize=(16, 5.4))
ring = np.linspace(0, 2 * np.pi, 300)
stops = np.radians(np.arange(0, 360, 45))
titles = ("(a) rigid rotation: the vane turns once per lap\n(omega_z = 2 Omega)",
          "(b) potential vortex: the vane never turns\n(omega_z = 0 for R > 0)")
for ax, rigid, title in zip(axes[:2], (True, False), titles):
    ax.plot(np.cos(ring), np.sin(ring), color=GRAY, lw=0.8)
    ax.annotate("", xy=(np.cos(1.32), np.sin(1.32)), xytext=(np.cos(1.05), np.sin(1.05)),
                arrowprops=dict(arrowstyle="->", color=GRAY, lw=1.2))
    for p in stops:
        draw_cross(ax, np.cos(p), np.sin(p), p if rigid else 0.0)   # vane angle = rotation since the start
    ax.text(1.0, -0.33, "start", ha="center", fontsize=8, color=GRAY)
    ax.plot(0, 0, "+", color=GRAY, ms=9)
    ax.set(xlim=(-1.5, 1.5), ylim=(-1.5, 1.5), aspect="equal", xticks=[], yticks=[], title=title)
axes[0].plot([], [], color=BLUE, lw=2.5, label="arm that starts radial")
axes[0].plot([], [], color=ORANGE, lw=2.5, label="arm that starts tangential")
axes[0].legend(fontsize=8, loc="lower left")

ax = axes[2]
for R0 in (0.5, 1.0, 1.5, 2.0):
    ax.plot(R0 * np.cos(ring), R0 * np.sin(ring), color=GRAY, lw=0.6)
ax.plot([-1, 1, 1, -1, -1], [-1, -1, 1, 1, -1], color=BLUE, lw=2,
        label=f"loop around the axis: C = {C_around:.2f} Gamma")
ax.plot([0.6, 1.6, 1.6, 0.6, 0.6], [0.6, 0.6, 1.6, 1.6, 0.6], color=ORANGE, lw=2,
        label=f"loop away from the axis: C = {abs(C_away):.2f}")
ax.plot(0, 0, "o", color=GRAY, ms=7)
ax.annotate("all the vorticity is here:\nomega_z = Gamma delta(x) delta(y)", xy=(0, 0), xytext=(-2.2, -2.15),
            fontsize=8, color=GRAY, arrowprops=dict(arrowstyle="->", color=GRAY, lw=0.8))
ax.set(xlim=(-2.4, 2.4), ylim=(-2.4, 2.4), aspect="equal", xticks=[], yticks=[],
       title="(c) circulation: Gamma around the axis,\nzero around a loop that misses it")
ax.legend(fontsize=8, loc="upper left")
fig.tight_layout()
fig.savefig(FIG_DIR / "ch00_vortex_vane.png", dpi=200, bbox_inches="tight")
plt.show()

## Summary

| Q | Key tool | Answer |
|---|---|---|
| 1 | $D/Dt = \partial/\partial t + u\,\partial/\partial x$ | Not necessarily: $DT/Dt = u\,\partial T/\partial x$ |
| 2 | Characteristics | $f = e^{-(x-3t)^2}$, moving right at speed 3 |
| 3 | Chain rule | $(\partial/\partial t)_x = \partial/\partial\tau - U\,\partial/\partial\xi$; $\;(\partial/\partial x)_t = \partial/\partial\xi$ |
| 4 | Dimensions; trial exponent | $\eta = x/\sqrt{4Dt}$; an error function |
| 5 | Mixed partials commute; $\varepsilon_{ijk}$ is antisymmetric | Both are zero |
| 6 | Book-keeping on a small box | $\oint_A\mathbf q\cdot\mathbf n\,dA = \int_V\nabla\cdot\mathbf q\,dV$ |
| 7 | Sifting, substitution, units | $e^2$; $\;1/3$; $\;\mathrm{m^{-1}}$ |
| 8 | Scale factors | $a^2\sin\theta\,d\theta\,d\varphi$; $\;\mathbf e_R$ turns with $\varphi$ |
| 9 | Derivatives of unit vectors | Centripetal acceleration: geometry, not a new force |
| 10 | Vorticity versus circulation | Irrotational for $R \neq 0$; $\;\omega_z = \Gamma\,\delta(x)\,\delta(y)$ |

**How the questions connect.**

- Q1 → Q2 → Q3: one idea, a moving observer, written three ways ($D/Dt$, characteristics, a moving frame).
- Q4 and Q7: dimensions. No intrinsic length scale gives a similarity solution; $\delta(x)$ carries the inverse units of $x$.
- Q8 → Q9: unit vectors turn, and the centripetal and angular-momentum terms follow.
- Q5 and Q7 → Q10: gradients are irrotational, yet in a domain with a hole an irrotational flow can carry circulation, supplied by
  a delta-function vorticity on the axis.

*My notes:*